# Exercise: Building a Transformer Block

Welcome! In this exercise, you will implement a standard Transformer `Block`, a key component of models like GPT. You'll combine modules you've learned about—Causal Self-Attention and a Multi-Layer Perceptron (MLP)—with residual connections and Layer Normalization.

This is a "pre-norm" implementation, a common variant where Layer Normalization is applied *before* the main transformation in each sub-layer.

**In this exercise you will:**
-   Implement the first sub-layer, which combines Layer Normalization, Causal Self-Attention, and a residual ("skip") connection.
-   Implement the second sub-layer, which combines Layer Normalization, an MLP, and a residual connection.
-   Assemble the complete `forward` pass of the `Block` by calling the two sub-layers in sequence.

Let's get started!

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import dataclasses

# ==================================
# SETUP - DO NOT MODIFY THIS CELL
# ==================================
# This cell provides the necessary building blocks and configuration that your
# Transformer Block will use. You do not need to modify this code.

@dataclasses.dataclass
class GPTConfig:
    """Configuration for a simple GPT model."""
    block_size: int = 256
    vocab_size: int = 65
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384

class CausalSelfAttention(nn.Module):
    """
    A simplified Causal Self-Attention module.
    (This is a dummy implementation for the exercise).
    """
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # A single linear layer to project inputs
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # This is a highly simplified forward pass for exercise purposes.
        # It maintains the shape but does not perform real attention.
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        # In a real implementation, this would involve Q, K, V projections,
        # scaled dot-product attention, and masking.
        # Here, we just project and return to test the wiring of the Block.
        x_projected = self.c_proj(x)
        return x_projected

class MLP(nn.Module):
    """
    A simple Multi-Layer Perceptron.
    (This is a dummy implementation for the exercise).
    """
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

print("Setup complete. You can now proceed to the exercises.")

### The Transformer Block Architecture

A Transformer `Block` consists of two main sub-layers. Our goal is to implement the `forward` pass for the `Block` shown below.

<img src="https://i.imgur.com/P4Pq27j.png" width="450" />

Notice the two "Add & Norm" steps. These represent the residual connection (`+`) followed by Layer Normalization. However, in the pre-norm formulation we are building, the order is swapped: **Norm -> Attention/MLP -> Add**.

Let's implement each part step-by-step.

### Exercise 1: The Attention Sub-layer

Your first task is to implement the logic for the first sub-layer. This involves:
1.  Applying the first Layer Normalization (`self.ln_1`) to the input `x`.
2.  Passing the result through the Causal Self-Attention module (`self.attn`).
3.  Adding the original input `x` to the output of the attention module (this is the residual connection).

The formula is: `x = x + self.attn(self.ln_1(x))`

Fill in the `forward_attention_sublayer` method below.

**Hint:** The operations should be performed in the exact order described above.

In [ ]:
class Block(nn.Module):
    """A single Transformer block."""

    def __init__(self, config: GPTConfig):
        """Initializes the layers required for a Transformer Block."""
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward_attention_sublayer(self, x: torch.Tensor) -> torch.Tensor:
        """
        Implements the first sub-layer of the Transformer Block.
        This is the Causal Self-Attention part with LayerNorm and a residual connection.
        x_out = x + Attention(LayerNorm(x))

        Args:
            x (torch.Tensor): The input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: The output tensor of the same shape.
        """
        ### START CODE HERE ### (≈ 2 lines)
        normalized_x = self.ln_1(x)
        attention_output = self.attn(normalized_x)
        x = x + attention_output
        ### END CODE HERE ###
        return x

    # The next two methods will be implemented in the following exercises.
    def forward_mlp_sublayer(self, x: torch.Tensor) -> torch.Tensor:
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pass

In [ ]:
# ==================================
# TEST CELL FOR EXERCISE 1
# ==================================
torch.manual_seed(42)

# 1. Setup the configuration and model
config = GPTConfig(n_embd=8, n_head=2, block_size=16, vocab_size=32)
block = Block(config)
block.eval() # Set to evaluation mode

# 2. Create a dummy input tensor
B, T, C = 2, 4, 8 # Batch, Time, Channels (n_embd)
x_input = torch.randn(B, T, C)

# 3. Get the student's output
student_output = block.forward_attention_sublayer(x_input.clone()) # Use clone to avoid in-place modification issues

# 4. Calculate the expected output
# Manually apply the same operations to get the correct result
with torch.no_grad():
    expected_normalized = block.ln_1(x_input.clone())
    expected_attn_output = block.attn(expected_normalized)
    expected_output = x_input.clone() + expected_attn_output

# 5. Assert that the shapes and values are correct
assert student_output.shape == x_input.shape, f"Expected shape {x_input.shape}, but got {student_output.shape}"
assert torch.allclose(student_output, expected_output, atol=1e-6), "The output values are incorrect."

print("\033[92mAll tests passed for Exercise 1! Great job.")

### Exercise 2: The Feed-Forward Sub-layer

Excellent! Now for the second part of the `Block`. This sub-layer is responsible for computation and feature transformation.

The structure is nearly identical to the first sub-layer, but it uses the MLP instead of the attention module:
1.  Apply the second Layer Normalization (`self.ln_2`) to the input `x`.
2.  Pass the result through the MLP (`self.mlp`).
3.  Add the original input `x` to the output of the MLP (the second residual connection).

The formula is: `x = x + self.mlp(self.ln_2(x))`

Fill in the `forward_mlp_sublayer` method below.

In [ ]:
def forward_mlp_sublayer(self, x: torch.Tensor) -> torch.Tensor:
    """
    Implements the second sub-layer of the Transformer Block.
    This is the MLP part with LayerNorm and a residual connection.
    x_out = x + MLP(LayerNorm(x))

    Args:
        x (torch.Tensor): The input tensor of shape (B, T, C).

    Returns:
        torch.Tensor: The output tensor of the same shape.
    """
    ### START CODE HERE ### (≈ 2 lines)
    normalized_x = self.ln_2(x)
    mlp_output = self.mlp(normalized_x)
    x = x + mlp_output
    ### END CODE HERE ###
    return x

# We attach the implemented method to our Block class for testing
Block.forward_mlp_sublayer = forward_mlp_sublayer

In [ ]:
# ==================================
# TEST CELL FOR EXERCISE 2
# ==================================
torch.manual_seed(42)

# 1. Setup the configuration and model
config = GPTConfig(n_embd=8, n_head=2, block_size=16, vocab_size=32)
block = Block(config)
block.eval()

# 2. Create a dummy input tensor
B, T, C = 2, 4, 8
x_input = torch.randn(B, T, C)

# 3. Get the student's output
student_output = block.forward_mlp_sublayer(x_input.clone())

# 4. Calculate the expected output
with torch.no_grad():
    expected_normalized = block.ln_2(x_input.clone())
    expected_mlp_output = block.mlp(expected_normalized)
    expected_output = x_input.clone() + expected_mlp_output

# 5. Assert that the shapes and values are correct
assert student_output.shape == x_input.shape, f"Expected shape {x_input.shape}, but got {student_output.shape}"
assert torch.allclose(student_output, expected_output, atol=1e-6), "The output values are incorrect."

print("\033[92mAll tests passed for Exercise 2! You're doing great.")

### Exercise 3: Assembling the Full Block

You've successfully implemented both sub-layers. The final step is to put them together in the main `forward` method.

The logic is simple: the output of the attention sub-layer becomes the input to the MLP sub-layer.

**Your task:**
-   Implement the `forward` method by calling `self.forward_attention_sublayer` and `self.forward_mlp_sublayer` in the correct sequence.

In [ ]:
def forward(self, x: torch.Tensor) -> torch.Tensor:
    """
    The full forward pass for the Transformer Block.

    Args:
        x (torch.Tensor): The input tensor.

    Returns:
        torch.Tensor: The final output tensor of the block.
    """
    ### START CODE HERE ### (≈ 2 lines)
    x = self.forward_attention_sublayer(x)
    x = self.forward_mlp_sublayer(x)
    ### END CODE HERE ###
    return x

# Attach the final method to our Block class
Block.forward = forward

In [ ]:
# ==================================
# INTEGRATION TEST FOR EXERCISE 3
# ==================================
torch.manual_seed(42)

# 1. Setup the configuration and complete model
config = GPTConfig(n_embd=8, n_head=2, block_size=16, vocab_size=32)
block = Block(config)
block.eval()

# 2. Create a dummy input tensor
B, T, C = 2, 4, 8
x_input = torch.randn(B, T, C)

# 3. Get the student's final output from the full forward pass
# A call to `block(x)` is shorthand for `block.forward(x)`
final_output = block(x_input.clone())

# 4. Calculate the expected final output by manually chaining the steps
with torch.no_grad():
    # First sub-layer
    intermediate_x = x_input.clone() + block.attn(block.ln_1(x_input.clone()))
    # Second sub-layer
    expected_final_output = intermediate_x.clone() + block.mlp(block.ln_2(intermediate_x.clone()))

# 5. Assert that the final output is correct
assert final_output.shape == x_input.shape, f"Expected final shape {x_input.shape}, but got {final_output.shape}"
assert torch.allclose(final_output, expected_final_output, atol=1e-6), "The final output values are incorrect."

print("\033[92mAll tests passed! You have successfully implemented a full Transformer Block.")
print("\033[92mCongratulations on completing the exercise!")